# Chapter 16: Contextual Bandits - Code Examples

This notebook demonstrates the contextual bandit algorithms from Chapter 16:
1. LinUCB (Linear Upper Confidence Bound)
2. Thompson Sampling for Linear Models (LinTS)
3. Shared LinUCB with Arm Features
4. Neural-Linear Thompson Sampling (Conceptual)

## Setup: Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## Listing 16.1 — Minimal LinUCB Implementation

This is the exact code from the chapter. We'll test it in subsequent cells.

In [ ]:
import numpy as np

class LinUCB:
    def __init__(self, n_arms, n_features, alpha=1.0, ridge=1.0):
        self.n_arms = n_arms
        self.n_features = n_features
        self.alpha = alpha
        I = np.eye(n_features)
        self.A = [ridge * I.copy() for _ in range(n_arms)]  # dxd
        self.b = [np.zeros((n_features,)) for _ in range(n_arms)]  # d

    def _theta_and_inv(self, arm):
        A_inv = np.linalg.inv(self.A[arm])
        theta_hat = A_inv @ self.b[arm]
        return theta_hat, A_inv

    def select(self, X):  # X: (n_arms, n_features) contexts at time t
        scores = np.empty(self.n_arms)
        for a in range(self.n_arms):
            theta_hat, A_inv = self._theta_and_inv(a)
            x = X[a]
            mean = x @ theta_hat
            bonus = self.alpha * np.sqrt(x @ A_inv @ x)
            scores[a] = mean + bonus
        return int(np.argmax(scores))

    def update(self, arm, x, reward):
        x = x.reshape(-1, )
        self.A[arm] += np.outer(x, x)
        self.b[arm] += reward * x

### Test LinUCB with Synthetic Data

Create a simple scenario with 3 arms and personalized rewards based on user context.

In [ ]:
# Simulate contextual bandit scenario
# 3 arms, 5 features: [device_mobile, device_desktop, hour_of_day_normalized, user_history_score, season_summer]
n_arms = 3
n_features = 5

# True parameters for each arm (ground truth)
# Arm 0: prefers mobile users in morning with history
# Arm 1: prefers desktop users in evening
# Arm 2: prefers summer season users
true_theta = [
    np.array([0.5, -0.2, -0.3, 0.4, 0.1]),  # Arm 0
    np.array([-0.2, 0.6, 0.4, 0.2, -0.1]),  # Arm 1
    np.array([0.1, 0.1, 0.0, 0.3, 0.7])     # Arm 2
]

def generate_context():
    """Generate random user context."""
    device_mobile = np.random.rand() > 0.5
    device_desktop = 1 - device_mobile
    hour = np.random.rand()  # 0-1 normalized
    history = np.random.rand()  # 0-1 score
    summer = np.random.rand() > 0.7  # 30% chance
    return np.array([device_mobile, device_desktop, hour, history, summer], dtype=float)

def get_reward(arm, context, noise_std=0.1):
    """Generate reward for arm given context (with noise)."""
    true_reward = context @ true_theta[arm]
    noisy_reward = true_reward + np.random.normal(0, noise_std)
    return np.clip(noisy_reward, 0, 1)  # Clip to [0, 1]

# Initialize LinUCB
linucb = LinUCB(n_arms=n_arms, n_features=n_features, alpha=1.5, ridge=1.0)

# Run simulation
n_rounds = 500
cumulative_reward = 0
rewards_history = []
arm_selections = []

for t in range(n_rounds):
    # Generate context for this round
    context = generate_context()
    
    # Create context matrix (same context for all arms in this simple version)
    X = np.tile(context, (n_arms, 1))
    
    # Select arm using LinUCB
    chosen_arm = linucb.select(X)
    arm_selections.append(chosen_arm)
    
    # Get reward
    reward = get_reward(chosen_arm, context)
    cumulative_reward += reward
    rewards_history.append(cumulative_reward)
    
    # Update model
    linucb.update(chosen_arm, context, reward)

print(f"LinUCB Results after {n_rounds} rounds:")
print(f"Total cumulative reward: {cumulative_reward:.2f}")
print(f"Average reward per round: {cumulative_reward/n_rounds:.4f}")

# Count arm selections
arm_counts = np.bincount(arm_selections, minlength=n_arms)
print(f"\nArm selection counts: {arm_counts}")
print(f"Arm selection percentages: {100*arm_counts/n_rounds}")

### Visualize LinUCB Learning Progress

In [ ]:
# Plot cumulative reward over time
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(rewards_history, linewidth=2)
plt.xlabel('Round', fontsize=12)
plt.ylabel('Cumulative Reward', fontsize=12)
plt.title('LinUCB: Cumulative Reward Over Time', fontsize=14)
plt.grid(alpha=0.3)

# Plot arm selection over time (rolling window)
plt.subplot(1, 2, 2)
window_size = 50
for arm in range(n_arms):
    arm_mask = np.array(arm_selections) == arm
    rolling_avg = np.convolve(arm_mask, np.ones(window_size)/window_size, mode='valid')
    plt.plot(range(window_size-1, len(arm_selections)), rolling_avg, 
             label=f'Arm {arm}', linewidth=2)

plt.xlabel('Round', fontsize=12)
plt.ylabel('Selection Probability (Rolling Average)', fontsize=12)
plt.title(f'LinUCB: Arm Selection Rate (Window={window_size})', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal learned parameters (theta_hat) for each arm:")
for arm in range(n_arms):
    theta_hat, _ = linucb._theta_and_inv(arm)
    print(f"Arm {arm}: {theta_hat}")
    print(f"  True:  {true_theta[arm]}")
    print(f"  Error: {np.linalg.norm(theta_hat - true_theta[arm]):.4f}")

## Listing 16.2 — Thompson Sampling for Linear Models (Pseudocode)

This is the conceptual pseudocode from the chapter. We'll implement a runnable version.

In [ ]:
# Maintain A = λI + ∑ φφ^T and b = ∑ rφ over chosen arms (φ is the feature vector)
def select_arm(Phi_candidates, A, b, sigma2=1.0):
    A_inv = np.linalg.inv(A)
    w_hat = A_inv @ b
    # Thompson step: sample a plausible parameter vector
    w = np.random.multivariate_normal(mean=w_hat, cov=sigma2 * A_inv)
    scores = Phi_candidates @ w
    return int(np.argmax(scores))

def update(A, b, phi, r):
    A += np.outer(phi, phi)
    b += r * phi

### Test LinTS (Shared Parameter Version)

Implement and test Thompson Sampling with shared parameters across arms.

In [ ]:
class LinTS:
    """Thompson Sampling for Linear Models (Shared Parameters)."""
    
    def __init__(self, n_features, ridge=1.0, sigma2=1.0):
        self.n_features = n_features
        self.sigma2 = sigma2
        self.A = ridge * np.eye(n_features)
        self.b = np.zeros(n_features)
    
    def select(self, Phi_candidates):
        """Select arm using Thompson Sampling.
        
        Args:
            Phi_candidates: (n_arms, n_features) feature matrix for all arms
        
        Returns:
            chosen_arm: index of selected arm
        """
        A_inv = np.linalg.inv(self.A)
        w_hat = A_inv @ self.b
        
        # Thompson step: sample from posterior
        w_sample = np.random.multivariate_normal(mean=w_hat, cov=self.sigma2 * A_inv)
        
        # Compute scores for all arms
        scores = Phi_candidates @ w_sample
        return int(np.argmax(scores))
    
    def update(self, phi, reward):
        """Update posterior with observed reward."""
        self.A += np.outer(phi, phi)
        self.b += reward * phi

# Test LinTS with same scenario
# We need a shared reward model, so we'll use a single true theta
true_theta_shared = np.array([0.3, 0.2, 0.1, 0.4, 0.3])

def get_reward_shared(context, noise_std=0.1):
    """Generate reward given context (shared model)."""
    true_reward = context @ true_theta_shared
    noisy_reward = true_reward + np.random.normal(0, noise_std)
    return np.clip(noisy_reward, 0, 1)

# For this test, we'll use different arm features
def generate_arm_features(n_arms):
    """Generate distinct features for each arm."""
    # Each arm has base features that differentiate it
    arm_features = []
    for i in range(n_arms):
        # Create arm-specific feature signature
        features = np.random.rand(n_features) * 0.5 + 0.25 * i
        arm_features.append(features / np.linalg.norm(features))  # Normalize
    return np.array(arm_features)

# Initialize LinTS
lints = LinTS(n_features=n_features, ridge=1.0, sigma2=0.5)

# Generate fixed arm features
arm_features = generate_arm_features(n_arms)

# Run simulation
n_rounds = 500
cumulative_reward_ts = 0
rewards_history_ts = []
arm_selections_ts = []

for t in range(n_rounds):
    # Use arm features as candidate contexts
    Phi_candidates = arm_features
    
    # Select arm using LinTS
    chosen_arm = lints.select(Phi_candidates)
    arm_selections_ts.append(chosen_arm)
    
    # Get reward (based on chosen arm's features)
    reward = get_reward_shared(arm_features[chosen_arm])
    cumulative_reward_ts += reward
    rewards_history_ts.append(cumulative_reward_ts)
    
    # Update model
    lints.update(arm_features[chosen_arm], reward)

print(f"\nLinTS Results after {n_rounds} rounds:")
print(f"Total cumulative reward: {cumulative_reward_ts:.2f}")
print(f"Average reward per round: {cumulative_reward_ts/n_rounds:.4f}")

# Count arm selections
arm_counts_ts = np.bincount(arm_selections_ts, minlength=n_arms)
print(f"\nArm selection counts: {arm_counts_ts}")
print(f"Arm selection percentages: {100*arm_counts_ts/n_rounds}")

### Visualize LinTS Learning Progress

In [ ]:
# Plot cumulative reward over time
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(rewards_history_ts, linewidth=2, color='orange')
plt.xlabel('Round', fontsize=12)
plt.ylabel('Cumulative Reward', fontsize=12)
plt.title('LinTS: Cumulative Reward Over Time', fontsize=14)
plt.grid(alpha=0.3)

# Plot arm selection over time (rolling window)
plt.subplot(1, 2, 2)
window_size = 50
for arm in range(n_arms):
    arm_mask = np.array(arm_selections_ts) == arm
    rolling_avg = np.convolve(arm_mask, np.ones(window_size)/window_size, mode='valid')
    plt.plot(range(window_size-1, len(arm_selections_ts)), rolling_avg, 
             label=f'Arm {arm}', linewidth=2)

plt.xlabel('Round', fontsize=12)
plt.ylabel('Selection Probability (Rolling Average)', fontsize=12)
plt.title(f'LinTS: Arm Selection Rate (Window={window_size})', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Show learned parameters
A_inv = np.linalg.inv(lints.A)
w_hat = A_inv @ lints.b
print(f"\nFinal learned parameter (w_hat):")
print(w_hat)
print(f"True parameter:")
print(true_theta_shared)
print(f"Error: {np.linalg.norm(w_hat - true_theta_shared):.4f}")

## Listing 16.3 — Feature Encoding for Shared LinUCB Model

This is the exact code from the chapter showing recommendation system with shared LinUCB.

In [ ]:
import numpy as np

class SharedLinUCB:
    """Shared LinUCB with arm features encoded in context."""
    
    def __init__(self, n_user_features, n_arm_features, alpha=1.0, ridge=1.0):
        self.n_user_features = n_user_features
        self.n_arm_features = n_arm_features
        self.n_features = n_user_features + n_arm_features
        self.alpha = alpha
        
        # Single shared model across all arms
        self.A = ridge * np.eye(self.n_features)
        self.b = np.zeros(self.n_features)
    
    def _build_context(self, user_features, arm_features):
        """Concatenate user and arm features."""
        return np.concatenate([user_features, arm_features])
    
    def select(self, user_features, candidate_arms_features):
        """
        Select best arm from candidates.
        
        Args:
            user_features: shape (n_user_features,) - e.g., [is_mobile, hour_of_day, ...]
            candidate_arms_features: shape (n_candidates, n_arm_features)
                                    - e.g., each row is [category_tech, category_sports, 
                                            article_age_hours, ...]
        
        Returns:
            best_arm_idx: index into candidate_arms_features
        """
        A_inv = np.linalg.inv(self.A)
        theta_hat = A_inv @ self.b
        
        n_candidates = len(candidate_arms_features)
        scores = np.zeros(n_candidates)
        
        for i in range(n_candidates):
            # Build full context vector for this (user, arm) pair
            x = self._build_context(user_features, candidate_arms_features[i])
            
            # UCB score: predicted reward + uncertainty bonus
            mean = x @ theta_hat
            bonus = self.alpha * np.sqrt(x @ A_inv @ x)
            scores[i] = mean + bonus
        
        return int(np.argmax(scores))
    
    def update(self, user_features, arm_features, reward):
        """Update model after observing reward."""
        x = self._build_context(user_features, arm_features)
        self.A += np.outer(x, x)
        self.b += reward * x


# Example usage: Article recommendation
n_user_features = 5   # device_type (3 one-hot), hour_of_day (1), user_engagement_score (1)
n_arm_features = 4    # category (2 one-hot), article_age_hours (1), author_popularity (1)

bandit = SharedLinUCB(n_user_features, n_arm_features, alpha=1.5)

# User arrives: mobile user at 2pm with high engagement
user_context = np.array([
    0, 1, 0,    # device: [desktop, mobile, tablet] - mobile user
    14,         # hour_of_day: 2pm
    0.8         # user_engagement_score: 0-1 scale
])

# Candidate articles (retrieved by separate ranking system)
candidates = np.array([
    [1, 0, 2.5, 0.7],   # Article 1: tech category, 2.5 hours old, popular author
    [0, 1, 0.5, 0.3],   # Article 2: sports category, fresh, less popular author  
    [1, 0, 24, 0.9],    # Article 3: tech category, 1 day old, very popular author
])

# Select and serve
chosen_idx = bandit.select(user_context, candidates)
print(f"Serving article {chosen_idx + 1}")

# User clicks (reward = 1)
bandit.update(user_context, candidates[chosen_idx], reward=1)

### Test Shared LinUCB with Article Recommendation Scenario

Simulate a realistic article recommendation system with multiple users and articles.

In [ ]:
# Initialize Shared LinUCB for article recommendation
n_user_features = 5   # device (3 one-hot), hour_of_day, engagement_score
n_arm_features = 4    # category (2 one-hot), article_age, author_popularity

shared_bandit = SharedLinUCB(n_user_features, n_arm_features, alpha=1.5, ridge=1.0)

# Define ground truth: how features affect click probability
# [device_desktop, device_mobile, device_tablet, hour, engagement, 
#  cat_tech, cat_sports, age, popularity]
true_weights = np.array([
    0.1, 0.2, 0.05,  # Mobile users more engaged
    0.15,            # Hour matters (peak times)
    0.3,             # High engagement users click more
    0.4, 0.2,        # Tech articles perform better
    -0.1,            # Fresh articles preferred (negative age)
    0.25             # Popular authors help
])

def generate_user_context_realistic():
    """Generate realistic user context."""
    device_type = np.random.choice([0, 1, 2], p=[0.3, 0.5, 0.2])  # desktop, mobile, tablet
    device = np.zeros(3)
    device[device_type] = 1
    
    hour = np.random.randint(0, 24)  # Hour of day
    engagement = np.random.beta(2, 5)  # User engagement (skewed toward lower)
    
    return np.concatenate([device, [hour, engagement]])

def generate_article_features(n_articles=10):
    """Generate diverse article features."""
    articles = []
    for _ in range(n_articles):
        category = np.random.choice([0, 1], p=[0.6, 0.4])  # tech vs sports
        cat_features = np.zeros(2)
        cat_features[category] = 1
        
        age = np.random.exponential(5)  # Article age in hours (exponential)
        popularity = np.random.beta(2, 3)  # Author popularity
        
        articles.append(np.concatenate([cat_features, [age, popularity]]))
    
    return np.array(articles)

def compute_click_probability(user_context, article_features, true_weights):
    """Compute true click probability given context."""
    full_context = np.concatenate([user_context, article_features])
    logit = full_context @ true_weights
    prob = 1 / (1 + np.exp(-logit))  # Sigmoid
    return np.clip(prob, 0.01, 0.99)

# Simulate article recommendation system
n_rounds = 1000
n_candidate_articles = 5  # Show 5 article candidates per user

cumulative_reward_shared = 0
rewards_history_shared = []
article_selections = []

# Generate a pool of articles
article_pool = generate_article_features(n_articles=20)

for t in range(n_rounds):
    # New user arrives
    user_context = generate_user_context_realistic()
    
    # Retrieve candidate articles (random sample from pool)
    candidate_indices = np.random.choice(len(article_pool), size=n_candidate_articles, replace=False)
    candidate_articles = article_pool[candidate_indices]
    
    # Select article using Shared LinUCB
    chosen_idx = shared_bandit.select(user_context, candidate_articles)
    article_selections.append(chosen_idx)
    
    # Simulate click (binary reward based on true probability)
    true_prob = compute_click_probability(user_context, candidate_articles[chosen_idx], true_weights)
    reward = 1 if np.random.rand() < true_prob else 0
    
    cumulative_reward_shared += reward
    rewards_history_shared.append(cumulative_reward_shared)
    
    # Update model
    shared_bandit.update(user_context, candidate_articles[chosen_idx], reward)

print(f"\nShared LinUCB Results after {n_rounds} rounds:")
print(f"Total clicks (cumulative reward): {cumulative_reward_shared}")
print(f"Click-through rate (CTR): {100*cumulative_reward_shared/n_rounds:.2f}%")

# Show learned vs true weights
A_inv = np.linalg.inv(shared_bandit.A)
learned_weights = A_inv @ shared_bandit.b
print(f"\nLearned weights shape: {learned_weights.shape}")
print(f"True weights shape: {true_weights.shape}")
print(f"Weight estimation error: {np.linalg.norm(learned_weights - true_weights):.4f}")

### Visualize Shared LinUCB Performance

In [ ]:
# Plot results
plt.figure(figsize=(14, 5))

# Plot cumulative clicks over time
plt.subplot(1, 2, 1)
plt.plot(rewards_history_shared, linewidth=2, color='green')
plt.xlabel('Round (User Visit)', fontsize=12)
plt.ylabel('Cumulative Clicks', fontsize=12)
plt.title('Shared LinUCB: Cumulative Clicks Over Time', fontsize=14)
plt.grid(alpha=0.3)

# Plot CTR over time (rolling window)
plt.subplot(1, 2, 2)
window_size = 50
binary_rewards = [rewards_history_shared[i] - rewards_history_shared[i-1] if i > 0 else rewards_history_shared[0] 
                  for i in range(len(rewards_history_shared))]
rolling_ctr = np.convolve(binary_rewards, np.ones(window_size)/window_size, mode='valid')
plt.plot(range(window_size-1, len(binary_rewards)), 100*rolling_ctr, linewidth=2, color='green')
plt.xlabel('Round (User Visit)', fontsize=12)
plt.ylabel('CTR % (Rolling Average)', fontsize=12)
plt.title(f'Shared LinUCB: Click-Through Rate (Window={window_size})', fontsize=14)
plt.grid(alpha=0.3)
plt.axhline(y=100*cumulative_reward_shared/n_rounds, color='r', linestyle='--', 
            label=f'Overall CTR: {100*cumulative_reward_shared/n_rounds:.2f}%')
plt.legend()
plt.tight_layout()
plt.show()

# Compare learned vs true weights
print("\nWeight Comparison (First 9 features):")
print(f"{'Feature':<20} {'True':<10} {'Learned':<10} {'Diff':<10}")
print("-" * 50)
feature_names = ['device_desktop', 'device_mobile', 'device_tablet', 'hour', 'engagement',
                 'cat_tech', 'cat_sports', 'age', 'popularity']
for i, name in enumerate(feature_names):
    print(f"{name:<20} {true_weights[i]:>9.4f} {learned_weights[i]:>9.4f} {abs(true_weights[i]-learned_weights[i]):>9.4f}")

## Listing 16.4 — Neural-Linear Thompson Sampling (Conceptual)

This is pedagogical pseudocode from the chapter. It's not meant to be runnable as-is, but demonstrates the architecture. We include it for reference.

In [ ]:
# NOTE: This is conceptual pseudocode, not fully runnable
# It demonstrates the neural-linear architecture

import numpy as np
# import torch
# import torch.nn as nn

class NeuralLinearBandit:
    def __init__(self, input_dim, hidden_dim, feature_dim):
        # Neural network: context → features
        # self.feature_net = nn.Sequential(
        #     nn.Linear(input_dim, hidden_dim), nn.ReLU(),
        #     nn.Linear(hidden_dim, feature_dim)
        # )
        # self.optimizer = torch.optim.Adam(self.feature_net.parameters())
        
        # Bayesian linear head: A and b for posterior inference
        self.A = np.eye(feature_dim)
        self.b = np.zeros(feature_dim)
        self.replay_buffer = []  # Store (x, a, r) tuples
    
    def select(self, context, candidate_arms):
        """Thompson sampling with neural features."""
        # with torch.no_grad():
        #     phi = self.feature_net(context).numpy()  # Neural features (frozen)
        
        # Sample linear weights from posterior: w ~ N(μ, Σ)
        # A_inv = np.linalg.inv(self.A)
        # w_mean = A_inv @ self.b
        # w_sample = np.random.multivariate_normal(w_mean, A_inv)
        
        # Score each arm with sampled weights
        # scores = [w_sample @ phi for _ in candidate_arms]
        # return np.argmax(scores)
        pass
    
    def update(self, context, arm, reward):
        """Update Bayesian head immediately, store for periodic NN retraining."""
        # Immediate update: Update Bayesian linear head with current features
        # with torch.no_grad():
        #     phi = self.feature_net(context).numpy()  # Features from current NN
        # self.A += np.outer(phi, phi)  # Instant update to posterior
        # self.b += reward * phi
        
        # Store for later neural network retraining
        # self.replay_buffer.append((context, arm, reward))
        
        # Trigger neural network retraining every 100 observations
        # if len(self.replay_buffer) >= 100:
        #     self._train_batch()
        pass
            
    def _train_batch(self):
        """Batch training: update neural network via SGD, then update Bayesian head."""
        # See chapter for full implementation details
        pass

print("Neural-Linear Thompson Sampling is conceptual pseudocode.")
print("For production use, leverage frameworks like Vowpal Wabbit or PyTorch.")

## Comparison: LinUCB vs LinTS vs Shared LinUCB

Let's compare the three approaches on the same problem.

In [ ]:
# Setup common test scenario
n_arms_comp = 3
n_features_comp = 5
n_rounds_comp = 500

# True parameters for per-arm models
true_theta_comp = [
    np.array([0.5, -0.2, -0.3, 0.4, 0.1]),
    np.array([-0.2, 0.6, 0.4, 0.2, -0.1]),
    np.array([0.1, 0.1, 0.0, 0.3, 0.7])
]

def run_comparison_trial(algorithm_name, algorithm, n_rounds=500):
    """Run a trial and return cumulative rewards."""
    np.random.seed(42)  # For fair comparison
    
    rewards = []
    cumulative = 0
    
    for t in range(n_rounds):
        context = generate_context()
        
        if algorithm_name == "LinUCB":
            X = np.tile(context, (n_arms_comp, 1))
            chosen_arm = algorithm.select(X)
            reward = get_reward(chosen_arm, context)
            algorithm.update(chosen_arm, context, reward)
            
        elif algorithm_name == "LinTS":
            # For LinTS, use arm-specific features
            arm_features = np.array([context + 0.1*i for i in range(n_arms_comp)])
            chosen_arm = algorithm.select(arm_features)
            reward = get_reward_shared(arm_features[chosen_arm])
            algorithm.update(arm_features[chosen_arm], reward)
            
        elif algorithm_name == "SharedLinUCB":
            # Create arm features (one-hot encoding + context)
            arm_one_hot = np.eye(n_arms_comp)
            candidate_arms = np.concatenate([arm_one_hot, 
                                            np.tile(context[:2].reshape(1, -1), (n_arms_comp, 1))], 
                                           axis=1)
            chosen_arm = algorithm.select(context[-3:], candidate_arms)
            reward = get_reward(chosen_arm, context)
            algorithm.update(context[-3:], candidate_arms[chosen_arm], reward)
        
        cumulative += reward
        rewards.append(cumulative)
    
    return rewards

# Initialize algorithms
linucb_comp = LinUCB(n_arms=n_arms_comp, n_features=n_features_comp, alpha=1.5)
lints_comp = LinTS(n_features=n_features_comp, sigma2=0.5)
shared_comp = SharedLinUCB(n_user_features=3, n_arm_features=n_arms_comp+2, alpha=1.5)

# Run trials
print("Running comparison trials...")
rewards_linucb = run_comparison_trial("LinUCB", linucb_comp, n_rounds_comp)
rewards_lints = run_comparison_trial("LinTS", lints_comp, n_rounds_comp)
rewards_shared = run_comparison_trial("SharedLinUCB", shared_comp, n_rounds_comp)

# Plot comparison
plt.figure(figsize=(12, 6))
plt.plot(rewards_linucb, label='LinUCB', linewidth=2)
plt.plot(rewards_lints, label='LinTS', linewidth=2)
plt.plot(rewards_shared, label='Shared LinUCB', linewidth=2)
plt.xlabel('Round', fontsize=12)
plt.ylabel('Cumulative Reward', fontsize=12)
plt.title('Algorithm Comparison: Cumulative Reward Over Time', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal Cumulative Rewards after {n_rounds_comp} rounds:")
print(f"LinUCB: {rewards_linucb[-1]:.2f}")
print(f"LinTS: {rewards_lints[-1]:.2f}")
print(f"Shared LinUCB: {rewards_shared[-1]:.2f}")

print(f"\nAverage Reward per Round:")
print(f"LinUCB: {rewards_linucb[-1]/n_rounds_comp:.4f}")
print(f"LinTS: {rewards_lints[-1]/n_rounds_comp:.4f}")
print(f"Shared LinUCB: {rewards_shared[-1]/n_rounds_comp:.4f}")

## Summary

This notebook demonstrated the key contextual bandit algorithms from Chapter 16:

1. **LinUCB** - Per-arm parameters with UCB exploration strategy
2. **LinTS** - Shared parameters with Thompson Sampling
3. **Shared LinUCB** - Single model with arm features for cold-start resilience

### Key Takeaways:
- LinUCB provides deterministic exploration via confidence bounds
- LinTS uses probabilistic exploration via posterior sampling
- Shared models enable generalization and handle cold-start scenarios
- Choice depends on: data availability, cold-start frequency, computational constraints

### Production Considerations:
- Feature engineering is critical for performance
- Two-stage architectures (retrieval + bandit) for large catalogs
- Rich logging for offline evaluation (IPS/DR methods)
- Gradual rollout with A/B testing for safety